# Practical 3: Neural Network – Customer Churn Prediction

**Aim:** Predict whether a customer leaves (1) or stays (0) using a neural network.

**Dataset required:** `Churn_Modelling.csv` (place it in the same folder as this notebook).

If TensorFlow is not installed, run `!pip install tensorflow` in a notebook cell and restart the kernel if needed.

In [ ]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

### Step 2: Load and view the dataset

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")
print(df.head())

### Step 3: Select input and output
The original notebook selects feature columns 3 to 12 and the `Exited` target at column 13.

In [ ]:
X = df.iloc[:, 3:13].copy()
y = df.iloc[:, 13].copy()
print('Input shape:', X.shape)
print('Output shape:', y.shape)

### Step 4: Encode categorical columns
Gender is label-encoded. Geography is one-hot encoded so categories are represented as separate binary columns.

In [ ]:
le = LabelEncoder()
X['Gender'] = le.fit_transform(X['Gender'])
X = pd.get_dummies(X, columns=['Geography'], drop_first=True, dtype=int)

### Step 5: Split and standardize data
Fit the scaler using training data only, then apply it to the test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

### Step 6: Build the neural network
Two hidden layers contain 6 neurons each. Dropout is used to help reduce overfitting.

In [ ]:
model = Sequential()
model.add(Dense(6, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.3))
model.add(Dense(6, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.summary()

### Step 7: Compile and train the model

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)

### Step 8: Predict and evaluate
The model returns probabilities. A threshold of 0.5 converts them to class 0 or 1.

In [ ]:
probabilities = model.predict(X_test)
y_pred = (probabilities > 0.5).astype(int).ravel()

print('Final Accuracy:', accuracy_score(y_test, y_pred) * 100)
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

### Step 9: Plot accuracy

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

### Result / Output explanation
The output shows final accuracy, a confusion matrix and an accuracy graph. Class 0 means the customer stayed; class 1 means the customer left. Exact accuracy depends on the dataset and training run.